# RNN

In [1]:
#Imports
from joblib import Parallel, delayed
import torch
import numpy as np
import random

import pandas as pd
from matplotlib import pyplot as plt
from torch.optim.lr_scheduler import ExponentialLR
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (mean_squared_error,mean_absolute_error)
import time
import re
import os,math
import numpy as np
from torch import nn, Tensor
import torch.nn.functional as F
from torch.nn.modules.transformer import TransformerEncoderLayer
import pickle
import base64

### DATA LOADING

In [ ]:
df_global=pd.read_csv("Global_5min_interpolated_data.csv")
df_local=pd.read_csv("local__irradiance_5min_ambient.csv")

In [ ]:
#DATASET PREPARATION
Hours_24=288

data_global = list(df_global['shortwave_radiation_instant (W/m²)'])
data_local = list(df_local["Solar Radiation (W/m^2)"])

#14 days worth of data (4030 datapoints out of 4318(Total))
x = data_global[:-Hours_24] # Keeping the 15th day for testing
y = data_local[:-Hours_24] # Keeping the 15th day for testing

### USEFUL FUNCTIONS

In [9]:
def set_seed(seed):
    torch.manual_seed(seed)  # For CPU
    torch.cuda.manual_seed(seed)  # For GPU
    torch.cuda.manual_seed_all(seed)  # For all GPUs
    np.random.seed(seed)  # For NumPy
    random.seed(seed)  # For Python's random module
    torch.backends.cudnn.deterministic = True  # Ensures determinism
    torch.backends.cudnn.benchmark = False  # Disables optimizations that affect results

# Create sequences
def create_sequences(x, y, seq_len=144, output_len=2):
    X, Y = [], []
    for i in range(len(x) - seq_len - output_len + 1):
        X.append(x[i:i + seq_len])
        Y.append([x[i + seq_len], y[i + seq_len]])  # <--- global first, local second
    
    X = torch.tensor(np.array(X), dtype=torch.float32).reshape(-1, seq_len, 1)
    Y = torch.tensor(np.array(Y), dtype=torch.float32).reshape(-1, 2)
    return X,Y

#RNN Model
class RNNForecastingModel(nn.Module):
    def __init__(self, 
                 seq_len,
                 hidden_size_rnn=64,
                 bidirectional=False,
                 num_layers=1,
                 dropout_rnn=0.1,):
        
        super(RNNForecastingModel, self).__init__()

        self.seq_len = seq_len
        self.hidden_size_rnn = hidden_size_rnn
        self.num_layers=num_layers
        self.dropout_rnn=dropout_rnn
        
        #If you only have 1 layer (num_layers=1), there are no "inter-layer" connections to apply dropout to
        dropout_rnn_actual = dropout_rnn if num_layers > 1 else 0.0
        
    
        # RNN layer
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_size_rnn,
            num_layers=num_layers,
            dropout=dropout_rnn_actual,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        direction_factor = 2 if bidirectional else 1
        self.outlayer = nn.Linear(seq_len * direction_factor * hidden_size_rnn, 2)

    def forward(self, x):
        
        # x: (batch_size, seq_len, 1)
        out, _ = self.rnn(x)                # (batch_size, seq_len, hidden_size)
        out = out.reshape(out.size(0), -1)   # Flatten to (batch_size, seq_len * hidden_size)
        return self.outlayer(out)            # Direct output (batch_size, 2)
    


### Hyperparameters

result_file='Results_Table/Trialllll.csv'

seq_len=[24,48,96,144,288,384]
batch_size=[48,96,144]
learning_rate=[0.05,0.001,0.005,0.00001,0.00005]
epochs=[50,100,200,300,400,500]
hidden_size_rnn=[100,200,288]
num_layers=[1,2,3]
dropout_rnn=[0.1]


bidirectional=True

### Training Loop

In [ ]:
#Training loop

execution_time = []
results = [] 
models_csv=[]

run_number=1

for i in range(len(seq_len)):
    for j in range(len(batch_size)):
        for k in range(len(epochs)):  
            for m in range(len(learning_rate)):
                for p in range(len(dropout_rnn)):
                    for d in range(len(hidden_size_rnn)):
                        for a in range (len(num_layers)):

                            
                            #Setting a Random Seed
                            seed= random.randint(0, 2**32 - 1)
                            set_seed(seed) 
                                
                            #Creating Sequences of Input Data
                            X_train, y_train = create_sequences(x, y, seq_len=seq_len[i])
                            print("X_train, y_train: ", X_train.shape, y_train.shape)

                            try:
                                #Initialize Model
                                # device = torch.device("cuda")
                                device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


                                start_time = time.time()

                                EPOCHS = epochs[k]  
                                BATCH_SIZE = batch_size[j]  
                                LEARNING_RATE = learning_rate[m]


                                model = RNNForecastingModel(seq_len=seq_len[i], hidden_size_rnn=hidden_size_rnn[d],
                                                     dropout_rnn=dropout_rnn[p],
                                                     num_layers=num_layers[a], 
                                                     bidirectional=bidirectional).to(device)


                                model.train()
                                optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
                                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)
                                # scheduler = ExponentialLR(optimizer, gamma=0.9)
                                criterion = torch.nn.HuberLoss() 

                                # Define worker_init_fn for reproducibility
                                def worker_init_fn(worker_id):
                                    np.random.seed(seed + worker_id)  # Ensure each worker gets a different seed


                                train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,worker_init_fn=worker_init_fn)

                                # Create a list to store losses for this combination of seq_len and batch_size
                                current_loss_list = []

                                for epoch in range(EPOCHS):
                                    epoch_loss = 0.0
                                    for X_batch, y_batch in train_loader:
                                        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                                        optimizer.zero_grad()
                                        outputs = model(X_batch)
                                #         print("Output Shape: ", outputs.shape)
                                #         print("y_batch: ",y_batch.shape)
                                        loss = criterion(outputs, y_batch)
                                        loss.backward()
                                        optimizer.step()
                                        epoch_loss += loss.item()
                                    scheduler.step(epoch_loss)
                                    current_loss_list.append(loss.item())

    #                                     print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(train_loader):.4f}")


                                end_time = time.time()

                                #Execution Time
                                exec_time = end_time - start_time
                                execution_time.append(exec_time)
                                print(f"{run_number}_Completed Seq Len {seq_len[i]}, Batch Size {BATCH_SIZE},Epoch Size {EPOCHS},Learning Rate {LEARNING_RATE}, in {end_time - start_time:.2f} seconds")



                                #Predict the next day using test global data ---
                                model.eval()
                                predicted_local = []

                                # Start with the last known global sequence
                                current_global_seq = list(x[-seq_len[i]:]) 

                                for _ in range(Hours_24):  # predict 288 steps ahead
                                    input_tensor = torch.tensor(np.array(current_global_seq).reshape(1, -1, 1),dtype=torch.float32).to(device) 
                                    with torch.no_grad():
                                        output = model(input_tensor)
                                        pred_global,pred_local = output[0].cpu().numpy()
                                        
                                    predicted_local.append(pred_local)
                                    current_global_seq.append(pred_global)
                                    current_global_seq = current_global_seq[-seq_len[i]:]


                                # Compute RMSE
                                y_true = np.array(df_local["Solar Radiation (W/m^2)"][-288:])
                                y_pred = np.array(predicted_local)

                                # RMSE calculation
                                rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
                                print(f"Test RMSE: {rmse:.4f} \n")


                                # Append results to the results list
                                results_dict={
                                    'run_number': run_number,
                                    'seed':seed,
                                    'Dropout':dropout_rnn[p],
                                    'num_layers_lstm':num_layers[a],
                                    'hidden_size_lstm':hidden_size_rnn[d],
                                    'seq_len': seq_len[i],
                                    'batch_size': BATCH_SIZE,
                                    'epochs': EPOCHS,
                                    'learning_rate':LEARNING_RATE,
                                    'execution_time': exec_time,
                                    'RMSE':rmse,
                                    'Predictions': y_pred.tolist(), 
                                    'Loss':current_loss_list}

                                results.append(results_dict) 

                                # Convert to DataFrame and save
                                results_df = pd.DataFrame([results_dict])

                                # If file doesn't exist, write with header. Else append without header.
                                if not os.path.exists(result_file):
                                    results_df.to_csv(result_file, index=False, mode='w', header=True)
                                else:
                                    results_df.to_csv(result_file, index=False, mode='a', header=False)

                                run_number=run_number+1
                                
                                
                            except Exception as e:
                                print(f"? Skipping Run {run_number} due to error: {e}")
                                continue

X_train, y_train:  torch.Size([3933, 96, 1]) torch.Size([3933, 2])
1_Completed Seq Len 96, Batch Size 96,Epoch Size 5,Learning Rate 0.001, in 1.19 seconds
Test RMSE: 228.0141 

X_train, y_train:  torch.Size([3933, 96, 1]) torch.Size([3933, 2])
2_Completed Seq Len 96, Batch Size 96,Epoch Size 10,Learning Rate 0.001, in 2.15 seconds
Test RMSE: 180.5408 



#### Note: The best model from the results_df would be the one with the lowest RMSE reported on the test dataset